In [2]:
from utils.function import get_gemini

print(get_gemini(" "))

Yes, I'm here! How can I help you?


In [3]:
response = get_gemini("""
    Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
    Generera bostadspriser, månadsavgifter, address, stad, boarea i jsonformat (ej markdown)

    Exempel:
            {
                "address": "Fågelvägen 5,
                "price_sek": 3000000,
                "city": "Göteborg",
                "monthly_fee": 4000,
                "area": 60
            }   
                   
    Ge mig en lista på 5 bostäder
""")

response

'[\n    {\n        "address": "Norr Mälarstrand 62, 2 tr",\n        "price_sek": 7500000,\n        "city": "Stockholm",\n        "monthly_fee": 3500,\n        "area": 70\n    },\n    {\n        "address": "Linnégatan 34, vån 3",\n        "price_sek": 4800000,\n        "city": "Göteborg",\n        "monthly_fee": 4500,\n        "area": 85\n    },\n    {\n        "address": "Davidshallsgatan 18, 1 tr",\n        "price_sek": 3200000,\n        "city": "Malmö",\n        "monthly_fee": 3800,\n        "area": 65\n    },\n    {\n        "address": "Svartbäcksgatan 22B",\n        "price_sek": 2500000,\n        "city": "Uppsala",\n        "monthly_fee": 3200,\n        "area": 50\n    },\n    {\n        "address": "Vasagatan 10A",\n        "price_sek": 1950000,\n        "city": "Västerås",\n        "monthly_fee": 2900,\n        "area": 45\n    }\n]'

In [4]:


from pydantic import BaseModel, Field
import json 

class Apartment(BaseModel):
    address: str 
    city: str 
    price_sek: int = Field(gt=1000000, lt = 8000000) 
    monthly_fee: int 
    area: int 

class ApartmentList(BaseModel):
    objects: list[Apartment]


apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments
    



ApartmentList(objects=[Apartment(address='Norr Mälarstrand 62, 2 tr', city='Stockholm', price_sek=7500000, monthly_fee=3500, area=70), Apartment(address='Linnégatan 34, vån 3', city='Göteborg', price_sek=4800000, monthly_fee=4500, area=85), Apartment(address='Davidshallsgatan 18, 1 tr', city='Malmö', price_sek=3200000, monthly_fee=3800, area=65), Apartment(address='Svartbäcksgatan 22B', city='Uppsala', price_sek=2500000, monthly_fee=3200, area=50), Apartment(address='Vasagatan 10A', city='Västerås', price_sek=1950000, monthly_fee=2900, area=45)])

### Get address, city, price, monthly_fee for the interval 4M - 8M

In [20]:

addresses = [
    {"address": a.address,"city": a.city,"price": a.price_sek,"fee": a.monthly_fee}
    for a in apartments.objects
    if 4000000 < a.price_sek < 8000000  
]

addresses



[{'address': 'Norr Mälarstrand 62, 2 tr',
  'city': 'Stockholm',
  'price': 7500000,
  'fee': 3500},
 {'address': 'Linnégatan 34, vån 3',
  'city': 'Göteborg',
  'price': 4800000,
  'fee': 4500}]

### Convert to df

In [49]:
import pandas as pd
df = pd.DataFrame(addresses)
df


,address,city,price,fee
0,"Norr Mälarstrand 62, 2 tr",Stockholm,7500000,3500
1,"Linnégatan 34, vån 3",Göteborg,4800000,4500


### Another way

In [ ]:
apartments.objects

[Apartment(address='Norr Mälarstrand 62, 2 tr', city='Stockholm', price_sek=7500000, monthly_fee=3500, area=70),
 Apartment(address='Linnégatan 34, vån 3', city='Göteborg', price_sek=4800000, monthly_fee=4500, area=85),
 Apartment(address='Davidshallsgatan 18, 1 tr', city='Malmö', price_sek=3200000, monthly_fee=3800, area=65),
 Apartment(address='Svartbäcksgatan 22B', city='Uppsala', price_sek=2500000, monthly_fee=3200, area=50),
 Apartment(address='Vasagatan 10A', city='Västerås', price_sek=1950000, monthly_fee=2900, area=45)]

In [39]:
address = [apart.address for apart in apartments.objects]
city = [apart.city for apart in apartments.objects]
price = [apart.price_sek for apart in apartments.objects]
fee = [apart.monthly_fee for apart in apartments.objects]
area = [apart.area for apart in apartments.objects]

new_df = pd.DataFrame({
"address": address,
"city": city,
"price": price,
"fee": fee,
"area": area
})
new_df



,address,city,price,fee,area
0,"Norr Mälarstrand 62, 2 tr",Stockholm,7500000,3500,70
1,"Linnégatan 34, vån 3",Göteborg,4800000,4500,85
2,"Davidshallsgatan 18, 1 tr",Malmö,3200000,3800,65
3,Svartbäcksgatan 22B,Uppsala,2500000,3200,50
4,Vasagatan 10A,Västerås,1950000,2900,45


In [25]:
type(apartments), apartments

(__main__.ApartmentList,
 ApartmentList(objects=[Apartment(address='Norr Mälarstrand 62, 2 tr', city='Stockholm', price_sek=7500000, monthly_fee=3500, area=70), Apartment(address='Linnégatan 34, vån 3', city='Göteborg', price_sek=4800000, monthly_fee=4500, area=85), Apartment(address='Davidshallsgatan 18, 1 tr', city='Malmö', price_sek=3200000, monthly_fee=3800, area=65), Apartment(address='Svartbäcksgatan 22B', city='Uppsala', price_sek=2500000, monthly_fee=3200, area=50), Apartment(address='Vasagatan 10A', city='Västerås', price_sek=1950000, monthly_fee=2900, area=45)]))

In [28]:
type(apartments.model_dump()), apartments.model_dump()

(dict,
 {'objects': [{'address': 'Norr Mälarstrand 62, 2 tr',
    'city': 'Stockholm',
    'price_sek': 7500000,
    'monthly_fee': 3500,
    'area': 70},
   {'address': 'Linnégatan 34, vån 3',
    'city': 'Göteborg',
    'price_sek': 4800000,
    'monthly_fee': 4500,
    'area': 85},
   {'address': 'Davidshallsgatan 18, 1 tr',
    'city': 'Malmö',
    'price_sek': 3200000,
    'monthly_fee': 3800,
    'area': 65},
   {'address': 'Svartbäcksgatan 22B',
    'city': 'Uppsala',
    'price_sek': 2500000,
    'monthly_fee': 3200,
    'area': 50},
   {'address': 'Vasagatan 10A',
    'city': 'Västerås',
    'price_sek': 1950000,
    'monthly_fee': 2900,
    'area': 45}]})

In [27]:
type(apartments.model_dump_json()), apartments.model_dump_json()


(str,
 '{"objects":[{"address":"Norr Mälarstrand 62, 2 tr","city":"Stockholm","price_sek":7500000,"monthly_fee":3500,"area":70},{"address":"Linnégatan 34, vån 3","city":"Göteborg","price_sek":4800000,"monthly_fee":4500,"area":85},{"address":"Davidshallsgatan 18, 1 tr","city":"Malmö","price_sek":3200000,"monthly_fee":3800,"area":65},{"address":"Svartbäcksgatan 22B","city":"Uppsala","price_sek":2500000,"monthly_fee":3200,"area":50},{"address":"Vasagatan 10A","city":"Västerås","price_sek":1950000,"monthly_fee":2900,"area":45}]}')

In [31]:
with open("apartments.json", "w", encoding="utf-8") as file: 
    file.write(apartments.model_dump_json(indent=3))

### Load to duckdb
- different ways: 

In [ ]:
import duckdb
with duckdb.connect("Housing.duckdb") as conn: 
    conn.execute("CREATE SCHEMA IF NOT EXISTS staging") #skapa schema

    conn.execute("CREATE OR REPLACE TABLE staging.apartment AS SELECT * FROM df")
    df_duckdb = conn.query("SELECT * FROM staging.apartment").df()
df_duckdb
# conn.execute("SELECT * FROM df").df()

# conn.close()


,address,city,price,fee
0,"Norr Mälarstrand 62, 2 tr",Stockholm,7500000,3500
1,"Linnégatan 34, vån 3",Göteborg,4800000,4500


### with dlt

In [ ]:
import duckdb
import dlt

@dlt.resource(write_disposition="replace")
def load_data():
    for d in df.to_dict(orient="records"): #detta behövs i ipynb, i en py kan du bara yielda df! 
        yield d
    
pipeline = dlt.pipeline(
    pipeline_name="apartments",
    destination="duckdb",
    dataset_name="staging",
    
)

load_info = pipeline.run(load_data(),table_name="apartment")
print(load_info)

Pipeline apartments load step completed in 0.26 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\Lukas\VSCode\ai_engineering_natali_harju\code-alongs\07a_pydantic_basics\apartments.duckdb location to store data
Load package 1757331635.5663786 is LOADED and contains no failed jobs
